# 00 — Environment check
GPU, CUDA, library versions. RTX 5080 = Blackwell (sm_120): needs a PyTorch wheel built for CUDA 12.8+ (`cu128`).
Saves `results/env/env_info.json` and `results/env/pip_freeze.txt`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
%matplotlib inline
print("project root:", ROOT)

project root: <repo>


In [2]:
import platform, torch
print("python      ", platform.python_version())
print("torch       ", torch.__version__, "| built for CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())
print("cuda avail. ", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available -> reinstall PyTorch from the cu128 index (README section 1)"
p = torch.cuda.get_device_properties(0)
cap = f"sm_{p.major}{p.minor}"
print("GPU         ", p.name, cap, f"{p.total_memory / 1024**3:.1f} GB")
print("arch list   ", torch.cuda.get_arch_list())
if cap not in torch.cuda.get_arch_list():
    print(f"WARNING: {cap} not in this wheel's arch list -> expect 'no kernel image' errors; install cu128 or newer")

python       3.10.11
torch        2.11.0+cu128 | built for CUDA 12.8 | cuDNN 91900
cuda avail.  True
GPU          NVIDIA GeForce RTX 5080 sm_120 15.9 GB
arch list    ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [3]:
# tiny CUDA workloads: matmul, bf16 autocast, cuDNN LSTM, Transformer layer
a = torch.randn(1024, 1024, device="cuda")
b = a @ a; torch.cuda.synchronize()
print("fp32 matmul OK", float(b.abs().mean()))
with torch.autocast("cuda", dtype=torch.bfloat16):
    c = a @ a
print("bf16 autocast OK", c.dtype, "| bf16 supported:", torch.cuda.is_bf16_supported())
lstm = torch.nn.LSTM(64, 128, 2, batch_first=True, bidirectional=True).cuda()
out, _ = lstm(torch.randn(4, 100, 64, device="cuda"))
print("cuDNN BiLSTM OK", tuple(out.shape))
enc = torch.nn.TransformerEncoderLayer(256, 4, 512, batch_first=True, norm_first=True).cuda()
print("Transformer layer OK", tuple(enc(torch.randn(4, 50, 256, device="cuda")).shape))

fp32 matmul OK 25.60063934326172
bf16 autocast OK torch.bfloat16 | bf16 supported: True
cuDNN BiLSTM OK (4, 100, 256)
Transformer layer OK (4, 50, 256)


In [4]:
import importlib
for m in ["numpy", "scipy", "pandas", "sklearn", "librosa", "soundfile", "soxr", "transformers",
          "matplotlib", "joblib", "yaml", "tabulate", "psutil"]:
    try:
        mod = importlib.import_module(m)
        print(f"{m:13s}", getattr(mod, "__version__", "ok"))
    except Exception as e:
        print(f"{m:13s} MISSING -> {e}")

numpy         2.2.6
scipy         1.15.3
pandas        2.3.3
sklearn       1.7.2
librosa       0.11.0
soundfile     0.14.0
soxr          1.1.0


<venv>/site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers  5.20.2
matplotlib    3.10.9
joblib        1.6.0
yaml          6.0.4
tabulate      0.10.0
psutil        7.2.2


In [5]:
import subprocess
from src import utils
info = utils.env_info()
utils.save_json(info, utils.RESULTS_DIR / "env" / "env_info.json")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(utils.ensure_dir(utils.RESULTS_DIR / "env") / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
info

{'os': 'Windows-10-10.0.26200-SP0',
 'python': '3.10.11',
 'cpu': 'AMD64 Family 26 Model 68 Stepping 0, AuthenticAMD',
 'cpu_count': 32,
 'numpy': '2.2.6',
 'scipy': '1.15.3',
 'pandas': '2.3.3',
 'sklearn': '1.7.2',
 'librosa': '0.11.0',
 'soundfile': '0.14.0',
 'soxr': '1.1.0',
 'transformers': '5.20.2',
 'matplotlib': '3.10.9',
 'joblib': '1.6.0',
 'yaml': '6.0.4',
 'ram_gb': 31.1,
 'torch': '2.11.0+cu128',
 'torch_cuda': '12.8',
 'cudnn': 91900,
 'cuda_available': True,
 'gpu': 'NVIDIA GeForce RTX 5080',
 'gpu_capability': 'sm_120',
 'gpu_mem_gb': 15.9,
 'torch_arch_list': ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120'],
 'nvidia_smi': 'NVIDIA GeForce RTX 5080, 621.14, 16303 MiB'}